# WWDC 2026 Python ハンズオン

このノートブックでは、WWDC 2026 で Apple が発表した Python 関連技術を体験します。

- Foundation Models SDK for Python（`apple-fm-sdk`）
- `coreai-torch`（PyTorch → Core AI 変換、参考構想）
- MLX Swift と Python MLX の対比

> 前提：macOS 27.0 以上を想定。SDK は今後変更になる可能性があります。

## 1. 環境セットアップ

必要なパッケージをインストールし、Swift / Xcode の確認方法も併記します。
実際に `apple-fm-sdk` や `coreai-torch` が公開されていない場合は、セルを **Markdown 表示用**に留めてください。

In [ ]:
# 必要なパッケージをインストール（公開状況により実行できない場合あり）
# !pip install apple-fm-sdk coreai-torch mlx

# インストール済みパッケージの確認
!pip list | grep -E "apple-fm-sdk|coreai|mlx|torch"

# Swift / Xcode の確認（ターミナル経由）
!swift --version
!xcode-select -p

## 2. `fm` CLI の基本的な使い方

macOS 27 以降に搭載される `fm` CLI をターミナルから呼び出します。
画像ファイル `Screenshot.png` がある場合のみ画像入力例を試してください。

In [ ]:
# fm CLI のバージョン確認
!fm --version

# 単発プロンプト
!fm respond "WWDC26 の Python 関連発表を3行でまとめて"

# Private Cloud Compute モデルを利用
!fm respond "複雑な正規表現を Swift で書いて" --model pcc

In [ ]:
# 画像入力例（Screenshot.png が存在する場合のみ）
from pathlib import Path
screenshot = Path("Screenshot.png")
if screenshot.exists():
    !fm respond "このスクリーンショットで何のアプリを使ってる？" --image Screenshot.png --model pcc
else:
    print("Screenshot.png が見つからないため、画像入力例をスキップします。")

# 構造化出力スキーマの生成
!fm schema object --name AppsIdentified --string app_names --array > schema.json
!cat schema.json

# 構造化出力を取得
if screenshot.exists():
    !fm respond "このスクリーンショットでアクティブなアプリは？" --image Screenshot.png --model pcc --schema schema.json
else:
    print("Screenshot.png がないため、構造化出力の実行もスキップします。")

## 3. Foundation Models SDK for Python — 基本の応答

`apple_fm_sdk` を使って、システム言語モデルからテキスト応答を取得します。


In [ ]:
import apple_fm_sdk as fm
import asyncio

async def basic_response():
    model = fm.SystemLanguageModel()
    available, reason = model.is_available()
    if not available:
        print(f"Language model is not available: {reason}")
        return

    session = fm.LanguageModelSession()
    response = await session.respond("Hello, Apple Intelligence!")
    print("=== Response ===")
    print(response.text)

# ノートブック内で非同期関数を実行
await basic_response()

## 4. Foundation Models SDK — ツール呼び出し

`fm.Tool` を継承して、モデルに外部関数を呼ばせる仕組みを作ります。
ここでは架空の「注文履歴取得ツール」を例にします。


In [ ]:
import apple_fm_sdk as fm

class Orders:
    @staticmethod
    async def load_last_orders(user_id: str, amount: int) -> str:
        # 実際には DB や API 呼び出しに置き換える
        return f"User {user_id}: 直近 {amount} 件の注文履歴（仮）"

class GetPastOrdersTool(fm.Tool):
    name = "get_past_orders"
    description = "過去の注文を取得する"

    @fm.generable
    class Arguments:
        number_orders: int = fm.guide("取得する件数")

    @property
    def arguments_schema(self) -> fm.GenerationSchema:
        return self.Arguments.generation_schema()

    async def call(self, args: fm.GeneratedContent) -> str:
        n = args.value(int, for_property="number_orders")
        user_id = "demo-user-123"
        return await Orders.load_last_orders(user_id=user_id, amount=n)

async def tool_call_demo():
    session = fm.LanguageModelSession(
        tools=[GetPastOrdersTool()],
    )
    response = await session.respond("直近 5 件の注文を教えて")
    print("=== Response ===")
    print(response.text)

await tool_call_demo()

## 5. Foundation Models SDK — 構造化出力

`@fm.generable` デコレータでクラスをマークし、モデルに決まった形の JSON を返させます。

In [ ]:
import apple_fm_sdk as fm

@fm.generable
class ItemSuggestion:
    name: str = fm.guide("商品名")
    category: str = fm.guide("カテゴリ")
    reason: str = fm.guide("推薦理由")

async def structured_output_demo():
    session = fm.LanguageModelSession()
    result = await session.respond(
        "おすすめの食材を1つ提案して",
        generating=ItemSuggestion
    )
    print("=== Structured Output ===")
    print(f"name: {result.name}")
    print(f"category: {result.category}")
    print(f"reason: {result.reason}")

await structured_output_demo()

## 6. `coreai-torch` で PyTorch モデルを Core AI 用に変換（参考構想）

> **注意**: `coreai-torch` は WWDC 2026 で発表された概念です。2026年7月時点では PyPI での公開・パッケージ名は未確定です。このセクションは発表内容に基づく参考構想としてご覧ください。

PyTorch で定義したモデルを `coreai_torch.convert()` で `.coreai` 形式に変換し、Swift / Core AI ランタイムへ渡します。

In [ ]:
import torch
import torch.nn as nn
import coreai.torch as coreai_torch  # 仮のパッケージ名

class MyPyTorchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = MyPyTorchModel()
model.eval()

sample_input = torch.randn(1, 3, 224, 224)

# PyTorch モデルを Core AI 用 AIProgram に変換
ai_program = coreai_torch.convert(
    model,
    example_inputs=(sample_input,),
    dynamic_dims={"batch": coreai_torch.Dim.unknown()}
)

# 保存
ai_program.save("my_model.coreai")
print("Saved my_model.coreai")

### Swift 側で推論するコード片

変換後の `my_model.coreai` を Swift / Core AI ランタイムで読み込むイメージ：

```swift
import CoreAI

let model = try AIGraph(contentsOf: URL(fileURLWithPath: "my_model.coreai"))
let result = try model.inference(input: imageTensor)
print(result)
```


## 7. MLX Swift と Python MLX の対比

Apple の MLX は元々 Python API があります。Swift API はその概念をミラーしています。


In [ ]:
# Python（MLX）での配列作成と演算
import mlx.core as mx

a = mx.array([1.0, 2.0, 3.0])
b = mx.array([4.0, 5.0, 6.0])
c = a + b
print(c)  # [5.0, 7.0, 9.0]

# 行列積の例
d = mx.array([[1.0, 2.0], [3.0, 4.0]])
e = mx.array([[5.0, 6.0], [7.0, 8.0]])
f = d @ e
print(f)

### Swift（MLX Swift）での対応コード

```swift
import MLX

let a = MLXArray([1.0, 2.0, 3.0])
let b = MLXArray([4.0, 5.0, 6.0])
let c = a + b
print(c)  // [5.0, 7.0, 9.0]

let d = MLXArray([[1.0, 2.0], [3.0, 4.0]])
let e = MLXArray([[5.0, 6.0], [7.0, 8.0]])
let f = d * e  // または matmul(d, e)
print(f)
```

### 対応関係まとめ

| Python | Swift |
|--------|-------|
| `import mlx.core as mx` | `import MLX` |
| `mx.array(...)` | `MLXArray(...)` |
| `+`, `@` 演算子 | `+`, `*` / `matmul` |
| `print(c)` | `print(c)` |


## まとめ

- **Python** は「実験・評価・変換」の言語
  - `fm` CLI + Python SDK でプロンプト評価パイプライン
  - `coreai-torch` で PyTorch モデルを Apple Silicon 向けに変換
  - MLX Python の知見を Swift に移植しやすい
- **Swift** は「製品実装」の言語
  - Core AI ランタイム、On-Device 推論、アプリ統合

次のステップ：

1. `examples/` のコードを自分のユースケースに改変する
2. WWDC26 公式セッション動画（Session 334/324/325/328）を視聴する
3. GitHub `apple/python-apple-fm-sdk` を watch する
